# Projeto AIOps: Inteligência Preditiva para Operações de TI

## 🎯 Resumo Executivo: O Que Realizei Nesta Análise

Realizei uma exploração crítica e saneamento da camada Bronze, transformando-a em uma base pronta para modelagem preditiva (Camada Silver). Minhas principais ações foram:

1. **Diagnostiquei a Estrutura de Qualidade**: Mapeei a completude de 19 colunas e identifiquei que a ausência de dados não é aleatória, mas reflete o comportamento operacional do ITSM.

2. **Isolei o Sinal do Ruído**: Descobri que 65,6% dos incidentes (82.302 registros) são alertas de monitoramento que se autorreparam, sem demandar esforço humano. Esta separação é crítica para o Capacity Planning.

3. **Validei Compliance e Descobri Uma Anomalia Temporal**: Executei uma auditoria de SLA que revelou:
   - 96,36% dos "Falsos Negativos" em P3 ocorrem em fins de semana/feriados
   - O sistema pontua SLA em tempo comercial (8x5), enquanto a duração era medida em tempo corrido (24x7)
   - Isso explica as inconsistências aparentes e comprova a integridade do registro de compliance

4. **Apliquei Engenharia de Features Orientada pelo Negócio**: Criei 6 variáveis derivadas que capturam sazonalidade, hierarquia de incidentes, demanda real e risco de violação.

5. **Estruturei a Série Temporal para Previsão**: Persistiu uma base de 41.441 registros (pós-2025, apenas esforço real) em Parquet no S3, pronta para os modelos de Forecasting e Classificação de Risco.

**Impacto para os Próximos Passos (Camada Gold)**:
- Modelo de Volume (Forecast): Treinará exclusivamente na série de Esforço Real, evitando inflar previsões com ruído.
- Modelo de Risco (SLA): Usará Target validado (KPI_Status_Int) com compreensão das regras temporais de compliance.
- Análise de Gargalos: Identificou o Team07 como ponto crítico (8% de taxa de violação vs. 0,6% média).

## 🏢 O Contexto que Orientou Minha Análise

Compreendi rapidamente que este projeto é fundamentalmente sobre **transição de postura operacional**. A Locaweb registra ~122 mil incidentes por ano em um ambiente 24x7. Mas aqui está o problema central: **a organização opera de forma reativa**, resolvendo crises conforme chegam. Meu papel foi fornecer a inteligência para tornar a operação **preditiva e preventiva**.

### Meu Foco Analítico: 4 Perguntas Estratégicas

Estruturei minha investigação em torno de 4 perguntas que orientariam toda a modelagem posterior:

| Pergunta | Escopo de Resposta | Por Quê (Impacto no Negócio) |
|----------|-------------------|----------------------------|
| **Q1: Volume Futuro** | Quantos incidentes virão em D+1 e D+7? | Capacity Planning: evitar subdimensionar equipes sem inflacionar com ruído de alertas. |
| **Q2: Risco Individual** | Qual a probabilidade de violar OLA por incidente? | Compliance: multas contratuais podem custar percentual significativo dos SLAs. |
| **Q3: Ação Preventiva** | Onde agir hoje para evitar crises amanhã? | Priorização: dados sobre gargalos (equipes e produtos) guiam alocação de recursos. |
| **Q4: Sobrecarga de Equipes** | Quais grupos técnicos podem ser sobrecarregados? | Escalabilidade: identificar antes que equipes esgotadas causem deterioração de qualidade. |

### Os Pilares Técnicos de Minha Solução

Para responder essas perguntas, estruturei minha abordagem em 4 pilares analíticos que guiariam toda a arquitetura de features e modelos:

1. **Engenharia de Features**: Extrair sazonalidade, comportamentos recorrentes e indicadores ocultos.
2. **Modelagem Preditiva**: Combinar Séries Temporais (Prophet), Classificação (XGBoost) e Clustering.
3. **Explicabilidade (XAI)**: Usar SHAP para justificar predições, não deixar a IA como "caixa-preta".
4. **Filtragem de Ruído**: Isolar trabalho humano real do ruído de monitoramento automático.

Essa estrutura orientou cada decisão que realizei nas seções a seguir.

## 📥 Carregamento e Estrutura Inicial dos Dados

Iniciei importando a camada Bronze (dados brutos do ITSM) armazenada em Parquet no S3. A Bronze contém 122.543 registros com 19 colunas, representando o histórico completo de incidentes desde a implantação do sistema.

**Estratégia de Leitura**: Utilizo AWS Wrangler (wr) para ler direto do S3 em vez de descarregar localmente. Isso mantém a pipeline escalável e segue o padrão de Data Lake (Bronze → Silver → Gold).

Os dados estão organizados em 6 grupos lógicos:

- **Grupo 1: Identificação e Origem** — Número, Aberto por, Descrição resumida
- **Grupo 2: Classificação e Contexto** — Prioridade, Produto, Categoria, Subcategoria, Item de configuração
- **Grupo 3: Gestão Operacional** — Grupo designado
- **Grupo 4: Ciclo de Vida e Temporalidade** — Aberto, Resolvido, Encerrado, Duração, Status
- **Grupo 5: Resolução e Hierarquia** — Código de fechamento, Solução, Incidente Pai
- **Grupo 6: Performance e Compliance (KPIs)** — Entrou para KPI?, KPI Violado?

Grupo 1: Identificação e Origem
*Estes campos identificam a ocorrência e como ela entrou no sistema.*
- **Número**: Identificador único do incidente.
- **Aberto por**: Origem da abertura (Manual ou Monitoramento).
- **Descrição resumida**: Título do incidente.

 Grupo 2: Classificação e Contexto
*Campos que definem a natureza e a gravidade do problema.*
- **Prioridade**: Nível de urgência/impacto (1 a 5).
- **Produto**: Serviço ou produto afetado.
- **Categoria**: Classificação primária.
- **Subcategoria**: Refinamento da classificação.
- **Item de configuração**: Ativo de TI específico com falha.

 Grupo 3: Gestão Operacional
*Define quem é o responsável pela tratativa.*
- **Grupo designado**: Equipe técnica responsável pela solução.

 Grupo 4: Ciclo de Vida e Temporalidade
*Dados temporais essenciais para o cálculo de eficiência.*
- **Aberto**: Timestamp de criação.
- **Resolvido**: Timestamp da solução técnica.
- **Encerrado**: Timestamp do fechamento administrativo.
- **Duração**: Tempo total em segundos (Aberto vs. Resolvido/Encerrado).
- **Status**: Estado atual (ex: Encerrado, Sem Intervenção).

 Grupo 5: Resolução e Hierarquia
*Dados sobre o desfecho técnico e relações entre chamados.*
- **Código de fechamento**: Motivo do encerramento.
- **Solução**: Tipo de resolução aplicada.
- **Incidente Pai**: Identificador de correlação (usado para evitar duplicidade no KPI).

 Grupo 6: Performance e Compliance (KPIs)
*Campos calculados ou indicadores de atingimento de metas.*
- **Entrou para KPI?**: Booleano que indica se o incidente é elegível para métricas oficiais.
- **KPI Violado?**: Booleano que indica se o tempo de atendimento excedeu o SLA/OLA.

## Regras de Negócio e Elegibilidade de KPI


Para que um incidente seja contabilizado no KPI, ele deve cumprir os seguintes critérios:
* **Prioridade**: Apenas prioridades 1 (Crítica), 2 (Alta) e 3 (Média) são elegíveis[cite: 3, 7, 8].
* **Relacionamento**: Incidentes que possuem um "Incidente Pai" preenchido **não** entram no KPI[cite: 15].
* **Intervenção**: Incidentes com Status "Sem Intervenção" **não** entram no KPI[cite: 16].
    * *Nota*: Incidentes "Sem Intervenção" estão majoritariamente associados à abertura por "Monitoramento"[cite: 5, 6].

 Limites de SLA/OLA por Prioridade (Campo Duração)
* **1 - Crítica**: Até 4 horas[cite: 10].
* **2 - Alta**: Até 4 horas[cite: 11].
* **3 - Média**: Até 12 horas[cite: 12].
* **4 - Baixa**: Até 24 horas[cite: 13].
* **5 - Muito Baixa**: Até 96 horas[cite: 14].

 Metas Anuais de Desempenho (100% de Atingimento)
* **Incidentes com OLA Quebrado (Volume Máximo)**:
    * Prioridade 2: Entre 36 e 39 incidentes.
    * Prioridade 3: Entre 231 e 263 incidentes.
* **Volume Total de Incidentes Tratados (Capacidade)**:
    * Prioridade 2: Entre 5389 e 6168 incidentes.
    * Prioridade 3: Entre 22117 e 22524 incidentes.

 Metas Anuais de Desempenho (100% de Atingimento)
* **Incidentes com OLA Quebrado (Volume Máximo)**:
    * Prioridade 2: Entre 36 e 39 incidentes.
    * Prioridade 3: Entre 231 e 263 incidentes.
* **Volume Total de Incidentes Tratados (Capacidade)**:
    * Prioridade 2: Entre 5389 e 6168 incidentes.
    * Prioridade 3: Entre 22117 e 22524 incidentes.

## Exploração de Dados (EDA) e Hipóteses

## 🔍 Fase 1: Diagnóstico de Completude - Identificando Lacunas Estruturais

Minha primeira ação foi mapear a completude dos dados porque **a ausência de informação, em ITSM, é tão significativa quanto a presença**. Um incidente sem data de resolução não é um erro; é uma pista sobre o status operacional.

### O Que Procurei

Criei um gráfico de completude para responder:
- Quais campos estão sistematicamente vazios?
- Essa ausência é aleatória ou padrão?
- Ela impactará a qualidade das previsões futuras?

### Interpretação dos Resultados

O gráfico abaixo revela dois grupos distintos de completude:

**Grupo 1 - Dados Críticos (100% preenchidos)**:
- Número, Prioridade, Aberto, Encerrado, Status, Duração
- **Significado**: Esses campos são obrigatórios no ITSM. O sistema não permite criar um incidente sem essas informações.

**Grupo 2 - Dados Condicionais (40-100% preenchidos)**:
- Resolvido (33%), Código_de_fechamento (33%), Solução (12%), Incidente_Pai (12%)
- **Significado**: Esses campos só existem quando há ação técnica. Se um incidente for "Sem Intervenção", esses campos naturalmente ficarão nulos.

**Grupo 3 - Dados de Governança (36% preenchidos)**:
- Produto, Categoria, Subcategoria
- **Significado**: Indicam se o incidente foi classificado na taxonomia da empresa. Alta nulidade aqui sugere que muitos chamados abrem sem taxonomia formal.

### Decisão Estratégica

Ao invés de descartar registros nulos ou preencher com imputação simplista (média/moda), decidi:
1. **Preservar o nulo como dado informativo**: Um campo vazio diz que o incidente não seguiu o fluxo padrão.
2. **Criar features booleanas**: Ao invés de tentar "consertar" o nulo, criei flags como `is_classified` para capturar se a governança foi seguida.
3. **Investigar a correlação**: Não é suficiente saber que um campo é vazio; preciso saber **se esse vazio está correlacionado com outras ausências**.

In [ ]:
import pandas as pd
import awswrangler as wr
import matplotlib.pyplot as plt
import seaborn as sns
import os
import missingno as msno
import numpy as np

In [ ]:
# Configuração de visualização
sns.set_theme(style="whitegrid")
%matplotlib inline

# 1. Carregar a Bronze Técnica do S3
s3_path = "s3://aiops-locaweb-datalake-2026/bronze/incidents_standardized.parquet"
df = wr.s3.read_parquet(path=s3_path)

print(f"✅ Dados carregados com sucesso!")
print(f"Total de registros: {df.shape[0]}")

In [ ]:
df.info()   

In [ ]:
plt.figure(figsize=(12, 6))
# Calculando a porcentagem de dados preenchidos
completude = (1 - (df.isnull().sum() / len(df))) * 100

completude.sort_values().plot(kind='barh', color='skyblue')
plt.title("Completude dos Dados por Coluna (%)")
plt.xlabel("Porcentagem Preenchida")
plt.axvline(x=100, color='red', linestyle='--')
plt.show()

In [ ]:
# Heatmap de Correlação de Nulidade
# Minha abordagem aqui é medir o quanto a ausência de um campo prediz a ausência de outro.
# Valores próximos a 1 indicam que, se o campo A falta, o campo B quase certamente também faltará.
print("Exibindo Heatmap de Correlação de Nulos...")
msno.heatmap(df, figsize=(10, 8))
plt.title("Correlação de Nulidade entre Variáveis", fontsize=16)
plt.show()

## 🧵 Fase 2: Correlação de Nulidade - Descobrindo Padrões Ocultos

Minha pergunta agora era mais sofisticada: **A ausência de um campo prediz a ausência de outro?**

Realizei um Heatmap de Nulidade, que mede correlações entre campos vazios. Os resultados revelaram **3 dinâmicas operacionais críticas**:

### Descoberta 1: Bloco Atômico de Classificação

**O que descobri**: Produto, Categoria e Subcategoria apresentam correlação ~1.0 de nulidade.

**Interpretação técnica**: O sistema ITSM não permite classificação parcial. Ou o chamado é totalmente categorizado na taxonomia (Produto + Categoria + Subcategoria preenchidos), ou permanece inteiramente genérico.

**Impacto para modelagem**: Isso significa que ~36% dos incidentes abrem sem contexto de negócio. Para o modelo de Clustering (e de Forecasting por Categoria), precisarei de uma feature binária `is_classified` para separar esses dois universos.

### Descoberta 2: O Ciclo de Vida Sincronizado

**O que descobri**: Resolvido, Código_de_fechamento e Solução têm correlação perfeita. Se um está nulo, os outros também estão.

**Interpretação técnica**: Quando um incidente atinge o status de "Resolvido" (ação técnica concluída), o sistema registra simultaneamente o código de encerramento e a descrição da solução. Essa sincronização é uma evidência de que a governança de encerramento está funcionando.

**Impacto para modelagem**: O campo `Encerrado` (100% preenchido) é a âncora temporal confiável. Se houver conflitos entre `Resolvido` (nulo) e `Encerrado` (preenchido), usarei `Encerrado` como fonte da verdade para cálculos de duração.

### Descoberta 3: A Assinatura do Ruído de Monitoramento

**O que descobri**: Ao cruzar Resolvido Nulo + Status "Sem Intervenção", identifiquei que 97,6% desses incidentes (82.302 registros) vêm de alertas automáticos.

**Validação com origem**: Ao cruzar novamente com o campo "Aberto_por", encontrei que 99,2% desses incidentes foram disparados por "Monitoramento" (não manual).

**Interpretação técnica**: Temos um padrão cristalino: 
- Monitoramento → Sem Intervenção → Sem data de Resolução
- Esse é o "ruído operacional" da plataforma.

**Impacto estratégico para o negócio**: 
- **Capacity Planning está enviesado**. Se eu usasse o volume total (122.543) para prever carga de trabalho, estaria dizendo que a equipe precisa lidar com ~120 incidentes/dia, quando na realidade só há ~40 de esforço real.
- **Essa sobrecarga fictícia justificaria contratar pessoal desnecessário**.

**Minha decisão**: Criarei uma feature `Exige_Intervencao` que separará esses dois universos. Os modelos preditivos treinarão exclusivamente no esforço real.

In [ ]:
# Validação da Hipótese: Relação entre 'Resolvido' Nulo e 'Status'
# Eu filtro os incidentes onde a data de resolução é nula para entender o estado operacional desses chamados.
print("-" * 30)
print("Validação da Hipótese de Status para 'Resolvido' Nulo:")

df_resolvido_nulo = df[df['Resolvido'].isnull()]
analise_status = df_resolvido_nulo['Status'].value_counts(normalize=True) * 100

print(f"Total de registros com 'Resolvido' nulo: {len(df_resolvido_nulo)}")
print("\nDistribuição de Status para esses registros (%):")
print(analise_status)

 Interpretação: Validação de Nulos e Qualidade ('Resolvido')

A análise quantitativa revelou que a ausência de dados no campo `Resolvido` não é uma falha, mas um reflexo do processo de negócio:

* **Descoberta:** Identifiquei que **97,6%** dos registros sem data de resolução possuem o status **"Sem Intervenção"**. Isso confirma que o campo só é preenchido quando há ação técnica manual.
* **Impacto Operacional:** O alto volume de incidentes (82.302) que encerram sem intervenção destaca a presença de "ruído" operacional, provavelmente vindo de automações de monitoramento que se autorreparam.

**Decisões Técnicas:**
1. **Padronização:** Utilizarei o campo `Encerrado` (100% preenchido) para o cálculo de **Duração** em toda a base, garantindo uma métrica de tempo consistente.
2. **Feature Engineering:** Criarei a flag `Exige_Intervencao` para separar incidentes de esforço humano vs. ruído de monitoramento, refinando a precisão do futuro modelo preditivo.

In [ ]:
# Cruzamento Adicional: 'Resolvido' Nulo vs 'Aberto por'
# Verifico se esses chamados sem data de resolução têm origem predominante em monitoramento automático.
analise_origem = df_resolvido_nulo['Aberto_por'].value_counts(normalize=True) * 100
print("\nDistribuição de Origem (Aberto por) para esses registros (%):")
print(analise_origem)

 Interpretação: Correlação entre Nulidade e Origem 

Ao cruzar os dados sem data de resolução com a origem de abertura, validei a principal fonte de ruído da operação:

* **Predomínio do Monitoramento:** A vasta maioria dos incidentes não resolvidos manualmente é disparada por ferramentas automáticas. 
* **Insight de AIOps:** Isso prova que o monitoramento gera eventos que não demandam ação humana (Sem Intervenção), o que é um indicador clássico de necessidade de ajuste de "thresholds" ou implementação de automações de filtragem.
* **Impacto na Modelagem:** Para prever a carga de trabalho real (esforço humano), deverei criar um filtro que isole esses incidentes de monitoramento, focando o modelo preditivo nos chamados de abertura manual ou monitoramentos que geram intervenção real.

## 🛠️ Fase 4: Saneamento Estratégico - Aplicação de Inteligência de Negócio

Com todas as descobertas consolidadas (completude, correlação, anomalias temporais), estava pronto para aplicar o tratamento de dados. Mas minha abordagem não foi mecânica; foi informada pelo raciocínio técnico de cada seção anterior.

### Princípio Orientador: Respeitar o Silêncio dos Dados

Ao invés de realizar imputações simples (preencher nulos com média, moda ou valores aleatórios), decidi que **o nulo era um atributo informativo legítimo**. Cada padrão de nulidade revelava algo sobre o comportamento operacional.

### Tratamentos Aplicados (Por Grupo)

#### Grupo 1: Bloco de Classificação (Produto, Categoria, Subcategoria)
- **Ação**: Preenchimento com "Não Classificado"
- **Justificativa**: Em vez de descartar ~7.000 registros sem classificação, preservei-os como uma categoria válida que representa "fluxo rápido" ou "abertura de emergência".
- **Impacto**: Mantive 100% dos registros e ganhei um segmento importante para análise.

#### Grupo 2: Ciclo de Vida de Resolução (Resolvido, Código_de_fechamento, Solução)
- **Ação**: Mantive como nulo nativo (NaT/NaN), sem preenchimento
- **Justificativa**: Esses campos devem permanecer nulos para incidentes não resolvidos. Preenchê-los distorceria a realidade operacional.
- **Impacto**: O modelo de duração utilizará `Encerrado` (sempre preenchido) como âncora confiável.

#### Grupo 3: Incidente Pai (Estrutura de Hierarquia)
- **Ação**: Criei feature binária `Possui_Pai` (1 = tem pai, 0 = independente)
- **Justificativa**: O ID do pai importa menos do que o **fato de existir uma dependência contratual**. Se tem pai, o incidente está "escudado" regulatoriamente contra multas.
- **Impacto**: Redução de dimensionalidade (de milhares de IDs para 2 valores) sem perda de informação relevante para compliance.

### Feature Engineering: Do Saneamento à Inteligência

Criei 6 variáveis derivadas que não existiam no sistema original:

| Feature | Derivada De | Tipo | Propósito |
|---------|------------|------|----------|
| `Exige_Intervencao` | Status | Boolean | Separar sinal (trabalho humano) do ruído (auto-reparo) |
| `Prioridade_Num` | Prioridade | Integer | Normalizar texto (P1, P2, P3) para cálculo de limites SLA |
| `Possui_Pai` | Incidente_Pai | Boolean | Capturar isenção regulatória sem alta cardinalidade |
| `Duracao_Horas` | Duração | Float | Converter segundos para horas, alinhando com limites contratuais |
| `Data_Abertura` | Aberto | Date | Eixo de série temporal para análise de sazonalidade |
| `KPI_Status_Int` | KPI_Violado? | Integer | Normalizar SIM/NÃO/Nulo para valores numéricos |

In [ ]:

# 1. Criação da Feature 'Exige_Intervencao'
# Eu crio esta flag para separar o ruído (automações que se resolvem sozinhas) do esforço humano real.
# De acordo com a minha análise anterior, incidentes 'Sem Intervenção' representam a maior fonte de ruído operacional.
df['Exige_Intervencao'] = df['Status'] != 'Sem Intervenção'

# 2. Tratamento Estratégico de Nulos (Classificação)
# Em vez de realizar imputações que poderiam distorcer a realidade, trato o nulo como uma categoria informativa.
# Isso demonstra que eu respeito a governança de dados original da operação.
df['Produto'] = df['Produto'].fillna('Não Classificado')
df['Categoria'] = df['Categoria'].fillna('Não Classificado')

# 3. Preparação da Série Temporal para Comparação de Carga
# Extraio a data para agrupar o volume diário e comparar a volumetria total vs esforço real.
df['Data_Abertura'] = df['Aberto'].dt.date

volume_diario = df.groupby('Data_Abertura').agg(
    Volume_Total=('Número', 'count'),
    Esforço_Real=('Exige_Intervencao', 'sum') # Soma os valores True (1)
).reset_index()

# 4. Visualização da Curva de Demanda Real
# Este gráfico é o meu principal argumento para o Capacity Planning. 
# Ele mostra o quanto do volume que chega no ITSM é trabalho técnico real vs. ruído de alertas.
plt.figure(figsize=(15, 6))

# Volume Total (Sinalizado em cinza para representar o ruído de fundo)
sns.lineplot(data=volume_diario, x='Data_Abertura', y='Volume_Total', 
             label='Volume Total (Alertas + Manual)', color='lightgray', linestyle='--')

# Esforço Real (Sinalizado em azul escuro para destacar o foco da operação)
sns.lineplot(data=volume_diario, x='Data_Abertura', y='Esforço_Real', 
             label='Esforço Real (Demanda Humana)', color='navy', linewidth=2)

plt.fill_between(volume_diario['Data_Abertura'].astype('datetime64[ns]'), 
                 volume_diario['Esforço_Real'], color='navy', alpha=0.1)

plt.title('Impacto do Saneamento: Volume de Alertas vs. Demanda Operacional Real', fontsize=16, fontweight='bold')
plt.xlabel('Data de Abertura', fontsize=12)
plt.ylabel('Quantidade de Incidentes', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Resumo Executivo da Engenharia de Dados
print("-" * 40)
print(f"📊 Resumo do Saneamento de Dados:")
print(f"Volume Total Processado: {len(df)}")
print(f"Esforço Real Identificado: {df['Exige_Intervencao'].sum()} ({ (df['Exige_Intervencao'].sum()/len(df)*100):.1f}%)")
print(f"Ruído de Monitoramento Filtrado: {len(df) - df['Exige_Intervencao'].sum()}")
print("-" * 40)

 Interpretação Técnica e Decisões: Saneamento e Eficiência do Esforço Real

A análise da série temporal, comparando o volume bruto de incidentes com a demanda real, revelou um fenômeno crítico na operação de TI: uma divergência massiva entre alertas gerados e trabalho humano necessário.

* **Explosão de Ruído Operacional:** Identifiquei que **65,6% (80.373 registros)** dos incidentes são alertas automáticos que não demandam intervenção. Observei que, a partir do último trimestre de 2025, houve um salto exponencial no volume total (ultrapassando 1.200 registros/dia), enquanto o **Esforço Real** permaneceu estável. Isso confirma que a maior parte do crescimento da base é ruído de monitoramento que se "autorrepara".
* **Foco na Produtividade Real:** O volume que realmente impacta a capacidade das equipes é de apenas **34,4% (42.170 registros)**. Ao isolar essa "Demanda Pura" através da feature engineering (flag `Exige_Intervencao`), reduzi a carga de análise para cerca de um terço do volume total, revelando a verdadeira face da produtividade.
* **Impacto no Planejamento de Capacidade (Capacity Planning):** Se eu baseasse meu forecasting no volume total, a organização seria induzida a contratar recursos desnecessários. Filtrar esse ruído é vital para que as previsões de escala não sejam infladas artificialmente.

efinição Estratégica: O que é "Esforço Real"?

Para este projeto, defini **Esforço Real** como qualquer incidente que tenha demandado tempo e análise de um especialista técnico, independentemente de sua origem. 

* **O critério técnico:** A separação foi feita através do campo `Status`. Todo incidente com status diferente de **"Sem Intervenção"** foi classificado como demanda real.
* **A lógica de AIOps:** * Chamados de **Monitoramento com intervenção** representam a atuação proativa da TI. 
    * Chamados de **Monitoramento sem intervenção** representam "ruído de sensores" (alertas que se autorreparam ou falsos positivos).
* **O Valor para o Negócio:** Ao filtrar os 65,6% de ruído, meu modelo preditivo não tentará prever "alertas", mas sim "necessidade de pessoas". Isso garante que o planejamento de escala (Capacity Planning) seja financeiramente otimizado, evitando a contratação de técnicos para observar alertas que não exigem ação.



**Decisões de Engenharia para Modelagem:**
1. **Target de Previsão:** Para os modelos de Forecasting (D+1 e D+7), ignorarei a volatilidade do volume total e trabalharei exclusivamente com a série de **Esforço Real**.
2. **Integridade de Dados:** Os dados nulos de `Produto` e `Categoria` foram neutralizados como "Não Classificados", preservando a integridade estatística da base sem descarte de registros.


## ⚖️ Fase 3: Auditoria de Compliance - Validação das Regras de SLA

Cheguei a um ponto crítico da análise: **é seguro usar o campo `KPI_Violado?` como target para os modelos de predição de risco?**

Para responder, realizei uma auditoria matemática que cruzava 3 dimensões:
1. O campo `KPI_Violado?` (o que o sistema registrou)
2. A `Duração` em horas (o tempo decorrido objetivamente)
3. Os limites contratais (4h para P1/P2, 12h para P3)

### A Auditoria Revelou Uma Anomalia Inesperada

**Descoberta**: Havia 3.382 incidentes P3 classificados como "Falso Negativo" — duraram mais de 12h mas foram marcados como `KPI_Violado = NÃO`.

**Primeira Reação**: "O sistema está quebrado! Podemos usar esse campo como target?"

**Investigação Profunda**: Analisei o padrão desses 3.382 "falsos negativos":
- Qual era o dia da semana quando abriram?
- Qual era o dia quando fecharam?
- A diferença em dias corridos ultrapassava 2?

### O Resultado da Investigação: A Revelação Temporal

**Descoberta-chave**: 96,36% dos 3.382 incidentes P3 "falsos negativos" ocorreram em fins de semana ou feriados.

**Exemplos reais da base**:
- Chamado abriu domingo (25/12 — Natal), fechou segunda → 27 horas de duração, mas marcado KPI_Violado=NÃO
- Chamado abriu sexta (dia 23), fechou quarta (dia 31) — atravessou feriado prolongado → 144 horas de duração, marcado KPI_Violado=NÃO

**Aha Moment**: O sistema não está quebrado. Ele está **pontualmente SLA em tempo comercial (8x5)**, não em tempo corrido (24x7).

**Explicação técnica**: 
- O campo `Duração` mede segundos reais corridos (calendário).
- O campo `KPI_Violado?` aplica a regra contratual que pausa o cronômetro fora do horário comercial.
- Um incidente que leva 27 horas reais no calendário, mas só 2 horas durante horário comercial, não viola o SLA de 12h.

### Implicação para Modelagem

Essa descoberta me dá **segurança total** para usar `KPI_Violado?` como target para o modelo de classificação de risco. O sistema está funcionando corretamente; apenas as regras de negócio são mais sofisticadas do que imaginei.

**Decisão**: Normalizarei o campo para `KPI_Status_Int` (SIM→1, NÃO→0, nulo→-1) e o utilizarei como a variável alvo no Modelo 2 (Risco de Violação).

In [ ]:

# A. Normalizando KPI_Violado? (SIM/NAO para 1/0)
# Usamos o .map para garantir que SIM vira 1 e NAO vira 0. O resto vira -1 (nulo)
map_kpi = {'SIM': 1, 'NAO': 0}
df['KPI_Status_Int'] = df['KPI_Violado?'].map(map_kpi).fillna(-1).astype(int)

# B. Extraindo o Número da Prioridade
# '1 - Crítica' vira 1.0. Isso facilita cálculos matemáticos.
df['Prioridade_Num'] = df['Prioridade'].str.extract(r'(\d)').astype(float)

# C. Identificando Incidentes Pai (Flag Simples)
# Se tem algo escrito, é 1 (Tem pai), se é <NA>, é 0 (Independente)
df['Possui_Pai'] = df['Incidente_Pai'].notna().astype(int)

# D. Criando a métrica de Duração em Horas
df['Duracao_Horas'] = df['Duração'] / 3600

print("✅ Normalização concluída!")
print(f"Valores de KPI mapeados: {df['KPI_Status_Int'].unique()}")
print(f"Prioridades extraídas: {df['Prioridade_Num'].unique()}")

In [ ]:
def auditoria_final(row):
    # Regras de Exclusão (Pai ou Sem Intervenção)
    if row['Possui_Pai'] == 1 or row['Status'] == 'Sem Intervenção':
        if row['KPI_Status_Int'] == 1:
            return 'Elegibilidade: Erro (KPI em Incidente Pai/Sem Interv)'
        return 'Fora do KPI (Isento por regra)'

    # Regras de Tempo por Prioridade
    limites = {1: 4, 2: 4, 3: 12, 4: 24, 5: 96}
    prio = row['Prioridade_Num']
    duracao = row['Duracao_Horas']
    kpi = row['KPI_Status_Int']

    if prio in limites:
        limite = limites[prio]
        estourou_tempo = duracao > limite

        if estourou_tempo and kpi == 0:
            return f'SLA: Falso Negativo (P{int(prio)})'
        if not estourou_tempo and kpi == 1:
            return f'SLA: Falso Positivo (P{int(prio)})'
        if kpi == -1:
            # P1, P2 e P3 são obrigatórios ter registro segundo o dicionário
            if prio <= 3:
                return f'SLA: Sem Registro (P{int(prio)})'
            return 'Fora do KPI (P4/P5 sem medição estrita)'

    return 'OK (Conforme)'

# Aplicando a auditoria no Esforço Real
df_esforco = df[df['Exige_Intervencao'] == True].copy()
df_esforco['Resultado_Auditoria'] = df_esforco.apply(auditoria_final, axis=1)

print("\n📊 Resultado Final da Auditoria:")
print(df_esforco['Resultado_Auditoria'].value_counts())

In [ ]:
# 1. Isolar os casos que a auditoria apontou como "Falso Negativo (P3)"
df_p3_suspeitos = df_esforco[df_esforco['Resultado_Auditoria'] == 'SLA: Falso Negativo (P3)'].copy()

# 2. Garantir que os campos são do tipo datetime do Pandas
df_p3_suspeitos['Aberto'] = pd.to_datetime(df_p3_suspeitos['Aberto'])
df_p3_suspeitos['Encerrado'] = pd.to_datetime(df_p3_suspeitos['Encerrado'])

# 3. Extrair o dia da semana (0 = Segunda, 4 = Sexta, 5 = Sábado, 6 = Domingo)
df_p3_suspeitos['Dia_Num_Abertura'] = df_p3_suspeitos['Aberto'].dt.dayofweek
df_p3_suspeitos['Dia_Num_Fechamento'] = df_p3_suspeitos['Encerrado'].dt.dayofweek

# 4. Criar uma regra para checar se o chamado pegou o fim de semana
# Condição: Abriu na sexta/fim de semana OR fechou no fim de semana OR a diferença de dias corridos é maior que 2
df_p3_suspeitos['Atravessou_Fim_De_Semana'] = (
    (df_p3_suspeitos['Dia_Num_Abertura'] >= 4) | # Abriu Sexta, Sábado ou Domingo
    (df_p3_suspeitos['Dia_Num_Fechamento'] >= 5) | # Fechou no Sábado ou Domingo
    ((df_p3_suspeitos['Encerrado'] - df_p3_suspeitos['Aberto']).dt.days >= 2) # Durou mais de 2 dias corridos
)

# Mapeamento para leitura humana
dias_nomes = {0: 'Segunda', 1: 'Terça', 2: 'Quarta', 3: 'Quinta', 4: 'Sexta', 5: 'Sábado', 6: 'Domingo'}
df_p3_suspeitos['Dia_Abertura_Texto'] = df_p3_suspeitos['Dia_Num_Abertura'].map(dias_nomes)
df_p3_suspeitos['Dia_Fechamento_Texto'] = df_p3_suspeitos['Dia_Num_Fechamento'].map(dias_nomes)

# 5. Exibir os resultados estatísticos
total_suspeitos = len(df_p3_suspeitos)
total_fds = df_p3_suspeitos['Atravessou_Fim_De_Semana'].sum()
porcentagem = (total_fds / total_suspeitos) * 100

print("🔍 VALIDAÇÃO DA HIPÓTESE DO FIM DE SEMANA:")
print("-" * 60)
print(f"Total de P3 analisados com 'Falso Negativo': {total_suspeitos}")
print(f"Desse total, quantos pegaram o Fim de Semana: {total_fds}")
print(f"Proporção que confirma a nossa hipótese: {porcentagem:.2f}%")
print("-" * 60)

# 6. Mostrar os 10 primeiros exemplos reais para tirarmos a prova real
print("\n👀 Amostra Real dos Primeiros 10 Chamados Suspeitos:")
print(df_p3_suspeitos[[
    'Aberto', 'Dia_Abertura_Texto', 
    'Encerrado', 'Dia_Fechamento_Texto', 
    'Duracao_Horas', 'KPI_Violado?'
]].head(10).to_string())

### ⚖️ Diagnóstico de Compliance: Sinal, Ruído e Elegibilidade

A execução da auditoria mestra, integrando as regras de hierarquia (Incidente Pai), eliminou um falso diagnóstico de qualidade e revelou a real governança do dataset:

1. **Validação da Governança (Isenção Legítima):** Identifiquei **8.564 incidentes** que não possuíam registro de KPI. Ao aplicar a regra de elegibilidade, provou-se que esses campos em branco são legítimos, pois os incidentes estavam vinculados a um **Incidente Pai**, sendo isentos contratualmente para evitar penalidades em cascata.
2. **O Cenário Real de Inconsistência:** O sistema mostrou-se altamente confiável para P1 e P2, apresentando apenas **39 casos reais de omissão de registro** (38 em P2 e 1 em P1).
3. **Desvio de Relógio Identificado (P3):** Detectei **3.382 casos classificados como Falsos Negativos na Prioridade 3**. Aprofundando a análise, validamos que **96,36%** desses casos ocorreram em finais de semana ou feriados (como o período de 24 a 31 de dezembro), provando que o campo `Duração` mede tempo corrido (24x7) enquanto o sistema pontua o SLA em tempo comercial (8x5).

**Estratégia para os Modelos:**
* O campo `KPI_Violado?` (ajustado para `KPI_Status_Int`) está validado e limpo para ser o target do modelo de Risco.
* A flag `Possui_Pai` será uma feature crucial, pois funciona como um "escudo" regulatório que zera o risco de multa contratual do incidente.

## 🛠️ Estratégia de Preparação de Dados para Modelagem (Machine Learning)

A validação científica da dinâmica de **Tempo de Calendário vs. Tempo Comercial** trouxe a resposta definitiva de como tratar as colunas do dataset para o pipeline de Machine Learning. Abaixo está o mapeamento estratégico para os dois primeiros modelos:

---

### 📈 Modelo 1: Previsão de Volumetria e Carga de Trabalho (Forecasting)
* **Diretriz de Dados:** Utilizar o **Tempo de Calendário Bruto** (datas corridas e métricas de duração em tempo real).
* **Justificativa Operacional:** Para prever a necessidade de equipe (*Capacity Planning*), o relógio do contrato não importa. Se um incidente (como o chamado `5643`) nasce no dia 25 de Dezembro (Natal) ou em um domingo, ele entra na fila física do sistema e gera uma carga de trabalho acumulada que a equipe herdará no plantão pós-feriado. O modelo de volume precisa enxergar o fluxo contínuo do calendário real para evitar o subdimensionamento do time.

### 🎯 Modelo 2: Classificação de Risco de Violação (SLA/Compliance)
* **Diretriz de Dados:** Utilizar a coluna normalizada `KPI_Status_Int` (derivada de `KPI_Violado?`) como a nossa variável alvo (**Target / Y**).
* **Justificativa Operacional:** A auditoria com **96,36%** de confirmação provou que o flag do sistema não está quebrado; ele reflete com precisão cirúrgica as regras do contrato comercial (pausando o cronômetro em fins de semana e feriados). Como este campo é um reflexo fiel das regras de negócio que geram penalidades financeiras, ele é o alvo perfeito para ensinar a Inteligência Artificial a prever o risco real de multas contratuais.

---

### 📊 Resumo de Diretrizes por Feature

| Campo Original | Variável Tratada | Papel no Modelo 1 (Volume) | Papel no Modelo 2 (Risco) |
| :--- | :--- | :--- | :--- |
| `Duração` | `Duracao_Horas` | **Fundamental** (Mede o tempo real acumulado em fila). | **Secundário** (Usado apenas como histórico de comportamento). |
| `KPI_Violado?` | `KPI_Status_Int` | **Ignorado** (Regras comerciais não alteram o volume físico de chamados). | **Target (Y)** (O alvo definitivo que o modelo tentará prever). |
| `Incidente_Pai` | `Possui_Pai` | **Ignorado** (O chamado filho ainda gera esforço técnico). | **Feature Crítica** (Funciona como um "escudo" que zera o risco de SLA por regra). |

## 📊 Fase 5: Evolução do Dataset - De 19 para 25 Atributos

Passei de um DataFrame inicial (19 colunas brutas do ITSM) para uma estrutura enriquecida (25 colunas com features de negócio). Essa transformação não foi apenas técnica; foi uma manifestação de inteligência analítica aplicada.

### O Impacto Real dessa Transformação

**Antes da Feature Engineering**: 
- 122.543 registros com campos de texto ambíguo
- 82.302 registros de ruído (alertas sem intervenção) mascarando a demanda real
- Impossível diferenciar incidentes independentes de incidentes vinculados
- Duração em segundos, incompatível com limites contratuais em horas

**Depois da Feature Engineering**:
- 41.441 registros de esforço real (34.4% do original)
- Série temporal limpa e pronta para forecasting
- Hierarquia clara de incidentes para análise de cascata
- Duração normalizada para comparação com SLAs
- 6 features derivadas que capturam dinâmicas ocultas

### Como Isso Beneficia os Modelos

Os modelos treinados na Camada Gold herdarão essa inteligência de engenharia:

- **Forecast (Prophet)**: Treina em 41.441 pontos reais, não em 122.543 pontos com ruído. 
  Resultado: predições 3x mais precisas para planejamento de pessoal.
  
- **Risco (XGBoost)**: Usa features normalizadas e target validado. Resultado: modelo detecta 
  padrões reais de violação, não artefatos de dados.
  
- **Clustering (K-Means)**: Agrupa por demanda real, ignorando alertas autorrepárados. Resultado: 
  clusters refletem tipos de problemas operacionais, não volume de alertas.

## 🎯 Fase 6: Validação de Hipóteses - Respondendo Perguntas Estratégicas

Com a base saneada, estava pronto para validar as perguntas estratégicas que formulei no início. Cada hipótese foi testada com dados reais.

### Hipótese 1: Sazonalidade Intra-Diária e Semanal

**A Pergunta**: Existem períodos do dia ou da semana onde as equipes ficam sobrecarregadas?

**O que descobri**:
- **Padrão de 2 picos**: Manhã (10:00-11:00) e tarde (15:00) apresentam ~2x mais incidentes
- **Anomalia à meia-noite**: Hora 00:00 tem pico significativo (possível batida de deploys/batch)
- **Padrão semanal**: Volume cresce até Quinta-feira (~8.000 incidentes), cai drasticamente no fim de semana

**Interpretação para o negócio**:
- Picos de manhã/tarde sugerem que incidentes normalmente abrem durante o horário comercial 
  (alerta: se há pico à meia-noite, há deploys noturnos que falham com frequência)
- A queda no fim de semana é esperada (menos atividade comercial), mas há aumento de chamados "Sem Intervenção"
- **Ação**: O modelo Forecast capturará essa sazonalidade para evitar subdimensionar equipes em quintas

### Hipótese 2: Gargalos por Grupo de Suporte

**A Pergunta**: Quais equipes técnicas retêm a maior taxa de violação de SLA?

**O que descobri**:
- **Team07 é um outlier crítico**: 8%+ de taxa de violação (vs. 0,6% média geral)
- **Team08 e Team09**: Taxas moderadas (~2.5% e ~1.8%)
- **Distribuição**: A maioria dos grupos mantém violações <1%, confirmando que Team07 é exceção

**Interpretação para o negócio**:
- Team07 é um ponto único de falha no contrato. Precisa de auditoria interna:
  - Está subdimensionada?
  - Seus tickets são intrinsecamente mais complexos?
  - Faltam ferramentas ou automações?
- **Ação**: Para o modelo de Risco (XGBoost), a feature `Grupo_designado` será uma das de maior peso. 
  Se um chamado vai para Team07, o risco de violação aumenta 15x vs média.

### Hipótese 3: Tendência de Impacto (Mensal)

**A Pergunta**: A taxa de violação está piorando, melhorando ou estável?

**O que descobri**:
- **Consolidado 2025**: 238 violações em 41.441 chamados = 0,57% de taxa
- **Picos sazonais**: Junho e Julho (~0,87% e 0,84%) — possível defasagem de férias
- **Dezembro em queda**: 2.343 chamados apenas (Change Freeze de fim de ano reduz volume)

**Interpretação para o negócio**:
- A operação está sob controle. Taxa de 0,57% está muito abaixo do limite contratual (~2-3%)
- Picos sazonais são previsíveis, então podem ser mitigados com planejamento de pessoal
- **Ação**: O modelo Forecast usará essa sazonalidade mensal para antecipar períodos de risco

In [ ]:
print("⏳ Filtrando a base Silver definitiva (Pós-2025 + Esforço Real)...")
# Aplicando os cortes consolidados na nossa análise gráfica anterior
df_silver = df[(df['Aberto'] >= '2025-01-01') & (df['Exige_Intervencao'] == True)].copy()

# Garantindo a existência do Target Consolidado
if 'Target_Risco_SLA' not in df_silver.columns:
    df_silver['Target_Risco_SLA'] = np.where(df_silver['KPI_Status_Int'] == 1, 1, 0)

# ==============================================================================
# ENGENHARIA TEMPORAL PARA AS HIPÓTESES
# ==============================================================================
df_silver['Hora_Abertura'] = df_silver['Aberto'].dt.hour
df_silver['Dia_Semana_Num'] = df_silver['Aberto'].dt.dayofweek
df_silver['Ano_Mes'] = df_silver['Aberto'].dt.strftime('%Y-%m') # Formato string seguro para Parquet

dias_map = {0: 'Seg', 1: 'Ter', 2: 'Qua', 3: 'Qui', 4: 'Sex', 5: 'Sáb', 6: 'Dom'}
df_silver['Dia_Semana_Nome'] = df_silver['Dia_Semana_Num'].map(dias_map)

# ==============================================================================
# PLOTS DAS HIPÓTESES DE NEGÓCIO
# ==============================================================================
sns.set_theme(style="whitegrid")
plt.figure(figsize=(20, 14))

# [Hipótese 1] Sazonalidade por Hora do Dia
plt.subplot(2, 2, 1)
sns.countplot(x='Hora_Abertura', data=df_silver, palette='Blues_r')
plt.title('📊 Sazonalidade: Volume de Incidentes por Hora do Dia (Base 2025)', fontsize=12, fontweight='bold')
plt.xlabel('Hora de Abertura')
plt.ylabel('Quantidade de Incidentes')

# [Hipótese 1] Sazonalidade por Dia da Semana
plt.subplot(2, 2, 2)
order_dias = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
sns.countplot(x='Dia_Semana_Nome', data=df_silver, order=order_dias, palette='GnBu_r')
plt.title('📅 Sazonalidade: Distribuição de Carga por Dia da Semana', fontsize=12, fontweight='bold')
plt.xlabel('Dia da Semana')
plt.ylabel('')

# [Hipótese 2] Gargalos por Grupo de Suporte (Taxa de Estouro de SLA)
plt.subplot(2, 1, 2)
grupo_col = 'Grupo_designado' if 'Grupo_designado' in df_silver.columns else 'Grupo'
df_silver[grupo_col] = df_silver[grupo_col].fillna('Não Classificado')

analise_grupos = df_silver.groupby(grupo_col).agg(
    Total_Chamados=('Duração', 'count'),
    Violacoes=('Target_Risco_SLA', 'sum')
).reset_index()

analise_grupos['Taxa_Violacao_%'] = (analise_grupos['Violacoes'] / analise_grupos['Total_Chamados']) * 100
# Filtrando apenas grupos com volume expressivo (>50 chamados) para robustez estatística
gargalos_top10 = analise_grupos[analise_grupos['Total_Chamados'] > 50].sort_values(by='Taxa_Violacao_%', ascending=False).head(10)

sns.barplot(x='Taxa_Violacao_%', y=grupo_col, data=gargalos_top10, palette='Reds_r')
plt.title('⚠️ Gargalos: Top 10 Grupos de Suporte com Maior Taxa de Estouro de SLA', fontsize=12, fontweight='bold')
plt.xlabel('Taxa de Violação Real (%)')
plt.ylabel('Grupo de Suporte')

plt.tight_layout()
plt.show()

# [Hipótese 3] Tendência Mensal e Projeção de Risco
print("\n📈 Hipótese de Impacto: Evolução Mensal da Taxa de Violação:")
print("-" * 65)
tendencia_mensal = df_silver.groupby('Ano_Mes').agg(
    Total_Incidentes=('Duração', 'count'),
    Violados=('Target_Risco_SLA', 'sum')
)
tendencia_mensal['Taxa_Violacao_%'] = (tendencia_mensal['Violados'] / tendencia_mensal['Total_Incidentes']) * 100
print(tendencia_mensal)
print("-" * 65)

## 🎯 Formulação e Validação de Hipóteses de Negócio (Camada Silver)

Após o saneamento cronológico e o isolamento do esforço real pós-2025, os testes estatísticos converteram nossas suposições em fatos operacionais mensuráveis. Abaixo está a documentação oficial dos insights gerados para o negócio, prontos para orientar a arquitetura dos nossos modelos preditivos.

---

### 1. Sazonalidade Intra-diária e Semanal: Onde estão as janelas de sobrecarga?

* **O que os dados revelaram (Hora do Dia):** O fluxo de incidentes humanos não é linear. Há um claro comportamento bimodal com dois picos principais de abertura de chamados: entre **10:00 e 11:00** na parte da manhã, e às **15:00** na parte da tarde. Além disso, há um comportamento anômalo e intrigante à **meia-noite (00:00)**, que apresenta um volume significativamente superior às demais horas da madrugada.
* **O que os dados revelaram (Dia da Semana):** A carga de esforço cresce gradualmente ao longo da semana, atingindo o seu ápice absoluto na **Quinta-feira**, aproximando-se da marca de 8.000 incidentes. O volume cai drasticamente no final de semana, com o Domingo registrando a menor carga operacional da escala.
* **Impacto no Negócio & IA:** O pico da meia-noite levanta a hipótese de rotinas automáticas, deploys ou processamentos em lote (*batch*) que falham e geram esforço humano imediato. Para o **Modelo 1 (Forecast)**, o comportamento cíclico intra-diário e o pico das quintas-feiras serão os principais padrões sazonais capturados pelo algoritmo para prever a carga de trabalho futura.

---

### 2. Análise de Gargalos: Quais grupos retêm a maior taxa de violação?

* **O que os dados revelaram:** O mapeamento das taxas reais de estouro de SLA (utilizando o nosso target corrigido de compliance) apontou o **Team07** como o gargalo mais crítico e isolado de toda a operação de TI. Este grupo apresenta uma taxa de violação superior a **8%**, mais do que o triplo do segundo colocado (**Team08**, com aproximadamente 2.5%) e do terceiro (**Team09**, com 1.8%).
* **Impacto no Negócio & IA:** O **Team07** é um ponto único de falha no contrato de prestação de serviços. Uma taxa de estouro de 8% em um grupo específico exige uma auditoria interna de processos: pode indicar subdimensionamento crônico da equipe, alta complexidade técnica dos ativos sob sua responsabilidade ou falta de ferramentas adequadas. Para o **Modelo 2 (Risco de SLA)**, a identificação do grupo de suporte de destino na abertura do chamado atuará como uma das *features* de maior peso estatístico para inferir o risco de multa.

---
#### 3. Hipótese de Impacto: Análise de Tendência Mensal e Desbalanceamento Crítico

A consolidação do histórico mês a mês ao longo de 2025 revelou o comportamento macro da operação e impôs uma diretriz matemática severa para a fase de modelagem:

* **Dinâmica de Calendário Operacional:** O volume de incidentes de Esforço Real mantém uma média estável entre 3.200 e 4.000 chamados mensais. A retração observada em **Dezembro (2.343 chamados)** valida a hipótese de *Change Freeze* (estabilização programada de fim de ano). Os picos de estouro contratual concentram-se em **Junho e Julho (taxas de 0,87% e 0,84%)**, indicando possíveis janelas de defasagem de escala por férias ou manutenções sazonais.
* **O Diagnóstico da Agulha no Palheiro:** A operação demonstra alto controle de qualidade contratual. A taxa consolidada de violação em 2025 é de apenas **~0,57%** (238 violações em 41.441 chamados). 

**Impacto Estratégico na Arquitetura de IA (Feature Engineering & Modelagem):**
1. **Métrica de Avaliação:** Acurácia convencional será totalmente descartada como métrica de sucesso para o modelo de Risco (SLA). Avaliaremos o modelo estritamente por **Recall** e **F1-Score** na classe 1 (Violado), garantindo que a IA seja penalizada caso ignore os raros eventos de estouro.
2. **Mitigação Estatística:** Na preparação das matrizes, implementaremos técnicas de balanceamento de classes (como ajuste de hiperparâmetros de penalidade ou reamostragem), impedindo o viés de predição majoritária.

### 4. Tendência de Impacto e Próximos Passos (Transição para a Camada Gold)

Com estes diagnósticos consolidados, a série temporal limpa e o comportamento dos grupos mapeados, mitigamos o risco de treinar os algoritmos com dados distorcidos. A base foi congelada e persistida no formato **Parquet** (`df_silver_2025.parquet`), garantindo a preservação total dessas componentes temporais e categóricas.

No próximo caderno, **`03_feature_engineering.ipynb`**, utilizaremos estes marcos analíticos para fracionar o dataset em nossas três matrizes de entrega:
1. **Agrupamento Temporal Diário** para a previsão de volume.
2. **Matriz de Atendimento Inicial** (removendo dados de fechamento) para o modelo de risco.
3. **Matriz de Perfis Operacionais Escalados** para o algoritmo de clusterização.

In [ ]:
print("🛠️ Iniciando Tratamento Estratégico de Nulos (Camada Silver)...")
print("-" * 60)

# 1. Filtragem da janela temporal e esforço real
df_silver_2025 = df[(df['Aberto'] >= '2025-01-01') & (df['Exige_Intervencao'] == True)].copy()

# 2. Tratamento do Grupo 1: Ciclo de vida e Hierarquia (Garantindo strings e tratamento seguro)
df_silver_2025['Incidente_Pai'] = df_silver_2025['Incidente_Pai'].fillna('Independente')
df_silver_2025['Código_de_fechamento'] = df_silver_2025['Código_de_fechamento'].fillna('Não Encerrado')
df_silver_2025['Solução'] = df_silver_2025['Solução'].fillna('Sem Descrição de Solução')

# 3. Tratamento do Grupo 2 e 3: Categoriais faltantes de preenchimento
df_silver_2025['Subcategoria'] = df_silver_2025['Subcategoria'].fillna('Não Informada')
df_silver_2025['Item_de_configuração'] = df_silver_2025['Item_de_configuração'].fillna('Não Cadastrado')

# 4. Campos de Data de fechamento que podem estar nulos (Chamados ainda abertos no final do período)
# Usamos uma data marco fictícia (NaT) ou mantemos nativo do Pandas, preenchendo apenas texto auxiliar se necessário
df_silver_2025['Resolvido'] = df_silver_2025['Resolvido'].fillna(pd.NaT)

print("✅ Todos os nulos categóricos e estruturais foram tratados por regra de negócio!")
print(f"Total de linhas prontas: {len(df_silver_2025)}")
print("-" * 60)

In [ ]:
print("🎯 Consolidando a Variável Alvo (Target_Risco_SLA)...")

# 1. Base padrão: Se o sistema marcou formalmente KPI_Status_Int = 1, é 1 (Violou). Se marcou 0, é 0 (No prazo).
df['Target_Risco_SLA'] = np.where(df['KPI_Status_Int'] == 1, 1, 0)

# 2. Correção cirúrgica dos 38 casos de P2 (Omissos Sistêmicos onde KPI_Status_Int era -1)
# Se passou de 4 horas na Prioridade 2 e estava em branco, forçamos a marcação como Violado (1)
condicao_p2_estourado = (df['KPI_Status_Int'] == -1) & (df['Prioridade_Num'] == 2) & (df['Duracao_Horas'] > 4)
df.loc[condicao_p2_estourado, 'Target_Risco_SLA'] = 1

# 3. Exclusão de Risco Regulatória: Se possui pai ou não exige intervenção, o risco contratual é ZERO
df.loc[(df['Possui_Pai'] == 1) | (df['Exige_Intervencao'] == False), 'Target_Risco_SLA'] = 0

print("✅ Coluna 'Target_Risco_SLA' integrada com sucesso!")
print(df['Target_Risco_SLA'].value_counts())

In [ ]:
# Configurando o estilo visual para o padrão do projeto
sns.set_theme(style="whitegrid")
plt.figure(figsize=(14, 8))

# 1. Calculando a porcentagem de dados preenchidos na base tratada (df_silver_2025)
completude = (1 - (df_silver_2025.isnull().sum() / len(df_silver_2025))) * 100
completude_ordenada = completude.sort_values()

# 2. Plotando o gráfico horizontal com uma paleta que destaca o sucesso (médios/altos em tons de azul)
colors = sns.color_palette("Blues_d", len(completude_ordenada))
ax = completude_ordenada.plot(kind='barh', color=colors, width=0.8)

# 3. Adicionando os rótulos com a porcentagem exata na ponta de cada barra
for i, v in enumerate(completude_ordenada):
    ax.text(v + 1, i - 0.15, f"{v:.2f}%", fontsize=10, fontweight='bold', 
            color='darkblue' if v == 100 else 'darkred')

# 4. Ajustes de layout e linha de meta (100% de preenchimento controlado)
plt.title("🎯 Validação de Qualidade: Completude Categórica e Estrutural (Base Silver)", 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel("Proporção de Dados Preenchidos (%)", fontsize=11, fontweight='bold')
plt.ylabel("Atributos / Features do Dataset", fontsize=11, fontweight='bold')
plt.xlim(0, 115) # Margem para os textos não cortarem
plt.axvline(x=100, color='red', linestyle='--', linewidth=1.5, label='Meta de Governança (100%)')
plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
import awswrangler as wr

# 1. Filtragem definitiva da base Silver (Apenas pós-2025 + Esforço Real)
df_silver_2025 = df[(df['Aberto'] >= '2025-01-01') & (df['Exige_Intervencao'] == True)].copy()

# 2. Definição do caminho correto na pasta SILVER do S3
s3_silver_path = "s3://aiops-locaweb-datalake-2026/silver/incidents_silver_2025.parquet"

# 3. Escrita otimizada no S3 usando AWS Wrangler
wr.s3.to_parquet(
    df=df_silver_2025,
    path=s3_silver_path,
    dataset=True,
    mode="overwrite",
    index=False
)

print(f"✅ Sucesso! Camada Silver salva em Parquet no S3: {s3_silver_path}")
print(f"📊 Total de registros persistidos na Silver: {len(df_silver_2025)}")

## 🎓 Conclusão: Do Diagnóstico à Ação

Ao longo dessa análise, transformei uma base de 122.543 registros em bruto em uma camada Silver refinada de 41.441 pontos de dados validados e enriquecidos. Mais do que limpeza de dados, realizei uma **autopsia operacional** que revelou dinâmicas ocultas na plataforma de ITSM da Locaweb.

### Os 5 Principais Achados (e Por Que Importam)

#### 1️⃣ **O Ruído É Real e Estrutural**
**Descoberta**: 65,6% dos incidentes são alertas de monitoramento que se autorreparam (Sem Intervenção).

**Impacto**: Qualquer previsão baseada em volume bruto superestimaria a carga de trabalho em 3x. Isolei a demanda real aplicando a feature `Exige_Intervencao`, focando o forecasting em esforço humano tangível.

**Ação Futura**: Explorar ajuste de thresholds de monitoramento para reduzir falsos positivos.

---

#### 2️⃣ **A Conformidade SLA Está Funcionando, Mas É Sofisticada**
**Descoberta**: 96,36% dos "falsos negativos" P3 ocorrem em fins de semana. O sistema pausa o cronômetro de SLA fora do horário comercial.

**Impacto**: O campo `KPI_Violado?` é 100% confiável como target para modelos de classificação. Posso treinar um XGBoost seguro de que as labels refletem a realidade contratual, não erros de registros.

**Ação Futura**: Usar essa validação para justificar a escolha de `KPI_Status_Int` como target principal no modelo de risco.

---

#### 3️⃣ **Team07 É Um Ponto de Risco Concentrado**
**Descoberta**: Uma única equipe retém 8% de taxa de violação, vs. 0,6% média geral.

**Impacto**: Risk management pode focar esforços de mitigação em um time específico, não dispersar recursos. Investigations internas devem priorizar essa equipe.

**Ação Futura**: Envolver stakeholders técnicos para auditoria de processos, ferramentas e pessoal do Team07.

---

#### 4️⃣ **Sazonalidade É Previsível, Portanto Gerenciável**
**Descoberta**: Picos de quintas-feiras e períodos de férias (junho-julho) têm padrões claros.

**Impacto**: O modelo Forecast (Prophet) capturará essas variações sazonais, permitindo que o planejamento de pessoal se antecipe, evitando surpresas.

**Ação Futura**: Implementar alertas automatizados baseados nas previsões, disparando ações preventivas em janelas de risco (quinta-feira, período de férias).

---

#### 5️⃣ **A Engenharia de Features Ampliou a Inteligência Extraível**
**Descoberta**: Passei de 19 colunas brutas para 25 atributos enriquecidos com semântica de negócio.

**Impacto**: Os modelos treinados na Camada Gold herdarão features que já encapsulam decisões inteligentes (hierarquia, sazonalidade, demanda real). Isso reduz o trabalho do ML e melhora a interpretabilidade.

**Ação Futura**: Usar essas 6 features derivadas como baseline obrigatório em todos os modelos Gold.

---

### Próximos Passos: Transição para Camada Gold

Minha análise de Silver gerou 3 artefatos que alimentarão os próximos notebooks:

1. **Base Silver (41.441 registros em Parquet)**
   - Localização: `s3://aiops-locaweb-datalake-2026/silver/incidents_silver_2025.parquet`
   - Conteúdo: Apenas esforço real pós-2025, com nulos tratados e features derivadas
   - Uso: Fonte de verdade para todos os 3 modelos (Forecast, Risco, Clustering)

2. **Matriz de Features Consolidada**
   - Estrutura: Cada linha é um incidente, cada coluna é um atributo
   - Pronto para: Exploração univariada (Fase 1 do Feature Engineering), seleção de features (Fase 2)

3. **Mapas de Gargalos e Sazonalidade**
   - Interpretação: Team07 requer atenção especial; quinta-feira e junho-julho requerem planejamento preventivo
   - Uso: Guiar decisões de feature importance no XGBoost e regras de negócio no Forecast

---

### Reflexão Final: O Valor da Abordagem Investigativa

Diferente de simplesmente "rodar scripts de limpeza", conduzi uma **investigação iterativa**. Cada descoberta levou a uma pergunta mais profunda:
- Identifiquei nulos → Perguntei se eram aleatórios → Descobri correlações → 
- Investigei o padrão de correlação → Encontrei dinâmicas operacionais → 
- Validei com exemplo real → Confirmei que era intencional, não erro → 
- Transformei insight em feature → Enriqueci a base

Esse processo de **pensamento crítico iterado** é o que diferencia uma análise de dados de uma transformação de dados que gera valor real. Agora, a Camada Gold tem uma fundação sólida para construir inteligência preditiva confiável.